# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [5]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [00:13<00:00,  4.49s/it]


In [6]:
len(deals)

30

In [7]:
deals[10].describe()

'Title: 11th-Gen. Apple iPad A16 128GB 11" Tablet (2025) for $282 + free shipping\nDetails: Best Buy offers the 11th-Gen. Apple iPad A16 128GB 11" Tablet (2025) for $299. That\'s $50 off and at least a buck less than other sellers. Shipping is free. Deal ends April 19.  Buy Now at Best Buy\nFeatures: A16 chip 11" 2360x1640 liquid retina display 128GB storage 12MP wide camera, 12MP Centre Stage camera WiFi 6 iPadOS Model: MD4A4LL/A\nURL: https://www.dealnews.com/products/Apple/11-th-Gen-Apple-iPad-A16-128-GB-11-Tablet-2025/498312.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [8]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [9]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [10]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Shokz OpenFit Open-Ear True Wireless Earbuds for $44 + free shipping
Details: Use promo code "VIPOUTLETAPR26" to drop the price. That makes for a really strong deal, it's $20 less than our previous mention and a total of $136 off now. Shipping is free. Coupon ends April 30 2026.    Buy Now at eBay
Features: DirectPitch audio up to 7 hours of listening on a single full charge Mo

In [11]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Shokz OpenFit open-ear true wireless earbuds feature DirectPitch audio delivery that keeps your ears open to ambient sound while providing up to 7 hours of continuous listening per charge. The T910-ST-BK-US model uses an open-fit design for comfort during workouts or commuting and includes the charging case for added portability. These are aimed at users who want situational awareness with wireless audio and long battery life in a lightweight package.', price=44.0, url='https://www.dealnews.com/products/Shokz/Shokz-Open-Fit-Open-Ear-True-Wireless-Earbuds/498168.html?iref=rss-c142'), Deal(product_description="LG QNED85A 65-inch 4K HDR QNED evo AI UHD Smart TV delivers 4K resolution with HDR10 support and a 120Hz refresh rate for smoother motion. It runs LG's webOS with built-in Alexa and Google Cast, and offers a versatile set of inputs including four HDMI and two USB ports. The 65QNED85AUA model targets users looking for a feature-rich sma

In [12]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Shokz OpenFit open-ear true wireless earbuds feature DirectPitch audio delivery that keeps your ears open to ambient sound while providing up to 7 hours of continuous listening per charge. The T910-ST-BK-US model uses an open-fit design for comfort during workouts or commuting and includes the charging case for added portability. These are aimed at users who want situational awareness with wireless audio and long battery life in a lightweight package.
44.0
https://www.dealnews.com/products/Shokz/Shokz-Open-Fit-Open-Ear-True-Wireless-Earbuds/498168.html?iref=rss-c142

LG QNED85A 65-inch 4K HDR QNED evo AI UHD Smart TV delivers 4K resolution with HDR10 support and a 120Hz refresh rate for smoother motion. It runs LG's webOS with built-in Alexa and Google Cast, and offers a versatile set of inputs including four HDMI and two USB ports. The 65QNED85AUA model targets users looking for a feature-rich smart TV with higher refresh rates and advanced upscaling driven by QNED evo AI.
599.0
https

In [13]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [14]:
from agents.scanner_agent import ScannerAgent

In [15]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [16]:
result

DealSelection(deals=[Deal(product_description='Shokz OpenFit Open-Ear True Wireless Earbuds (Model T910-ST-BK-US) are open-ear earbuds that sit outside the ear canal to allow ambient awareness while listening. They use DirectPitch audio technology and provide up to 7 hours of listening on a single full charge. The design targets users who want situational awareness during activities like running or commuting and includes the charging/carrying case for additional runtime.', price=44.0, url='https://www.dealnews.com/products/Shokz/Shokz-Open-Fit-Open-Ear-True-Wireless-Earbuds/498168.html?iref=rss-c142'), Deal(product_description='LG QNED85A Series 65QNED85AUA is a 65-inch 4K QNED evo AI UHD smart TV with HDR10 support and a 120Hz refresh rate for smoother motion. It runs WebOS and includes built-in voice assistants (Alexa and Google Cast), four HDMI inputs and two USB ports for connecting consoles and peripherals, and LG’s enhanced quantum dot + mini-LED picture processing for improved c

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [26]:
load_dotenv(override=True)

True

In [27]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [28]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with s


In [29]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [30]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [31]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [32]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
09:44:14 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= gpt-5.1; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5.1; provider = openai
09:44:16 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
